In [ ]:
from modules import *

In [ ]:
json_file = './all_trajectories.json'
time_seq, posi_seq = get_geolife_data(json_file)
print('len(time_seq), len(posi_seq): ', len(time_seq), len(posi_seq))
print('len(time_seq[0]), len(posi_seq[0]): ', len(time_seq[0]), len(posi_seq[0]))

new_time_seq = []
new_posi_seq = []

time_len_record = []
posi_len_record = []

for i in range(len(time_seq)):
    one_time_seq = time_seq[i]
    one_posi_seq = posi_seq[i]
    
    new_one_time_seq = []
    new_one_posi_seq = []
    
    for j in range(1, len(one_time_seq)):
        pre_time = one_time_seq[j-1]
        pre_lon, pre_lat = one_posi_seq[j-1]
        
        time = one_time_seq[j]
        lon, lat = one_posi_seq[j]
        
        new_one_time_seq.append(pre_time)
        new_one_posi_seq.append(one_posi_seq[j-1])
        
        if (time-pre_time) > 600.0:
            new_posi_seq.append(new_one_posi_seq)
            new_time_seq.append(new_one_time_seq)
            time_len_record.append(len(new_one_time_seq))
            posi_len_record.append(len(new_one_posi_seq))
            
            new_one_time_seq = []
            new_one_posi_seq = []

print('len(new_time_seq), len(new_posi_seq): ', len(new_time_seq), len(new_posi_seq))
print('len of time_len_record: ', len(time_len_record))
print('average len and min max len of trails: ', sum(posi_len_record)/len(posi_len_record), max(posi_len_record), min(posi_len_record))
print('new_time_seq[0][0]：', new_time_seq[0][0])

In [ ]:
lengths = [len(seq) for seq in new_time_seq]

print('average length of time sequences: ', sum(lengths)/len(lengths))

v_thre_time_seq = []
v_thre_posi_seq = []

for i in range(len(new_time_seq)):
    # v_thre_one_time_seq = []
    # v_thre_one_posi_seq = []
    count_abnormal = 0
    
    for j in range(1, len(new_time_seq[i])):
        pre_time = new_time_seq[i][j-1]
        pre_lon, pre_lat = new_posi_seq[i][j-1]
        
        time = new_time_seq[i][j]
        lon, lat = new_posi_seq[i][j]
        
        d_lon = lon - pre_lon
        d_lat = lat - pre_lat
        # ave_lat = (lat + pre_lat) / 2
        ave_lat = pre_lat
        
        lat_ave_rad = math.radians(ave_lat)
        d_lon_rad = math.radians(d_lon)
        d_lat_rad = math.radians(d_lat)
        
        dx = R * d_lon_rad * math.cos(lat_ave_rad)
        dy = R * d_lat_rad
        dt = time - pre_time
        
        v = math.sqrt(dx*dx + dy*dy) / (dt+1e-6)
        
        # if v > 200.0:
        #     print(f'异常点： {lon}, {lat}, {time}, {pre_lon}, {pre_lat}, {pre_time}, dx: {dx}, dy: {dy}, 速度：{v}, 时间: {dt}')
            
        if v > 45.0 or dt < 1.0:
            # print(f'异常点： {lon}, {lat}, {time}, {pre_lon}, {pre_lat}, {pre_time}, dx: {dx}, dy: {dy}, 速度：{v}')
            count_abnormal += 1
            break

            
    if count_abnormal < 1:
        v_thre_time_seq.append(new_time_seq[i])
        v_thre_posi_seq.append(new_posi_seq[i])

print('len(v_thre_time_seq), len(v_thre_posi_seq): ', len(v_thre_time_seq), len(v_thre_posi_seq))
sequence_lengths = np.array([len(seq) for seq in v_thre_time_seq])
print('average length of time sequences after velocity thresholding: ', np.mean(sequence_lengths))

sorted_indices = np.argsort(sequence_lengths)

sequence_lengths_sorted = sequence_lengths[sorted_indices]
sequence_lengths_sorted = sequence_lengths_sorted[sequence_lengths_sorted >= 5]
print(len(sequence_lengths_sorted))

print('sorted sequence lengths: ', sequence_lengths_sorted[:100])
idx_95 = int(0.95 * len(sequence_lengths_sorted))+1
length_95 = sequence_lengths_sorted[idx_95]
print('95 percent of sequences have length less than: ', length_95)


In [ ]:
import json

f = open('./trips_posi_seq.json', 'r')
trip_posi_seq = json.load(f)
f.close()

f = open('./trips_time_seq.json', 'r')
trip_time_seq = json.load(f)
f.close()

select_len = length_95

refined_time_seq = []
refined_posi_seq = []

for i in range(len(v_thre_time_seq)):
    len_seq = len(v_thre_time_seq[i])
    # print(f'Original length of trajectory {i}: {len_seq}')
    # 如果轨迹长度小于5（10倍下采样后的5），则直接丢弃该轨迹；如果轨迹长度小于select_len，则在末尾进行0填充；如果轨迹长度大于等于select_len，则直接截断到select_len。
    if len_seq < 5:
        continue
    elif len_seq < select_len:
        refined_time_seq.append(v_thre_time_seq[i])
        refined_posi_seq.append(v_thre_posi_seq[i])
    else:        
        refined_time_seq.append(v_thre_time_seq[i][:select_len])
        refined_posi_seq.append(v_thre_posi_seq[i][:select_len])
print('len(refined_time_seq), len(refined_posi_seq): ', len(refined_time_seq), len(refined_posi_seq))

lengths = np.array([len(seq) for seq in v_thre_time_seq])

print(lengths[:100])
print('np.mean(lengths): ', np.mean(lengths))

sorted_indices = np.argsort(lengths)
# np.argsort返回的是下标，即把lengths数组从小到大排序后对应的原始下标。sorted_indices[0]是最短轨迹的下标，sorted_indices[-1]是最长轨迹的下标。
lengths_sorted = lengths[sorted_indices]
lengths_sorted = lengths_sorted[lengths_sorted >= 5]

print('lengths_sorted: ', lengths_sorted[:10])

print(sorted_indices[0])
# print(norm_emb_time_seq[sorted_indices[0]])

# 去重后95%分位数对应的轨迹长度比不去重的更长，实际上重复长度的轨迹应该计算数量。去重后0.95分位数对应的轨迹长度是10349，而不去重的0.95分位数对应的轨迹长度是5204。
i = int(len(lengths_sorted)*0.95)+1
print(lengths_sorted[i])

emb_posi_seq, emb_posi_seq_d, emb_time_seq, start_time_emb, mask_ = construct_features(v_thre_time_seq, v_thre_posi_seq, select_len)
print('emb_posi_seq.shape: ', emb_posi_seq.shape)

In [ ]:
# print(len(mask_[0]))
print('mask_ shape: ', mask_.shape)
print('emb_posi_seq_d.shape', emb_posi_seq_d.shape)
# print('emb_time_seq.shape', emb_time_seq.shape)

all_dx = []
all_dy = []
all_log_dt = []
# all_log_dT = []
all_dt = []
# all_dT = []

trace_count = 0
for i in range(mask_.shape[0]):
    actual_len = len(mask_[i]) - mask_[i].sum().item()
    
    ds_dt_count = 0
    for j in range(actual_len):
        x = emb_posi_seq_d[i, j, 0]
        y = emb_posi_seq_d[i, j, 1]
        all_dx.append(x)
        all_dy.append(y)
        
        all_log_dt.append(np.log(emb_time_seq[i][j, 0]+1.0))
        # all_log_dT.append(np.log(abs_emb_time[i][j, 0]+1.0))
        all_dt.append(emb_time_seq[i][j, 0])
        
        if x*x+y*y <= 0.0 and emb_time_seq[i][j, 0] > 0.0:
            ds_dt_count += 1
            # print(f"位移为0但时间间隔大于0的点: 位移: ({x}, {y}) 米, 时间间隔: {emb_time_seq[i][j, 0]} 秒")
        # if emb_time_seq[i][j, 0]>600.0:
        #     print(f"异常时间间隔: {emb_time_seq[i][j, 0]} 秒, 位移: ({x}, {y}) 米")
        
        # if abs(x) > 4000 or abs(y) > 4000:
        #     print(f"异常位移: ({x}, {y}) 米, 时间间隔: {emb_time_seq[i][j, 0]} 秒")
            
    # all_dT.append(abs_emb_time[i][actual_len-1, 0])
    
    if ds_dt_count > 0:
        trace_count += 1

print(f"位移为0但时间间隔大于0的点数量: {ds_dt_count}, 轨迹数量: {trace_count}")

# print('max dT', np.max(np.array(all_dT)))
# print('min dT', np.min(np.array(all_dT)))
# print('mean dT', np.mean(np.array(all_dT)))
print('ds_dt_count: ', ds_dt_count)

print('len(all_dx), len(all_dy), len(all_log_dt): ', len(all_dx), len(all_dy), len(all_log_dt))

print(f"最大位移: {np.max(np.abs(all_dx))} 米")
print(f"99分位数: {np.percentile(np.abs(all_dx), 99)} 米")
print(f"大于 1000 米的点占比: {np.mean(np.abs(all_dx) > 1000) * 100:.4f}%")
print(f"平均步间时间间隔: {np.mean(np.array(all_dt))} 秒")
print(f"步间时间间隔标准差: {np.std(np.array(all_dt))}")

print(np.max(np.array(all_dt)))
print(np.sort(np.array(all_dt))[::-1][:10])

print(f"dt中位数时间间隔: {np.median(np.array(all_dt))} 秒")

ave_dx = np.mean(np.array(all_dx))
std_dx = np.std(np.array(all_dx))

ave_dy = np.mean(np.array(all_dy))
std_dy = np.std(np.array(all_dy))

ave_log_dt = np.mean(np.array(all_log_dt))
std_log_dt = np.std(np.array(all_log_dt))

# ave_log_dT = np.mean(np.array(all_log_dT))
# std_log_dT = np.std(np.array(all_log_dT))

print('ave_dx, std_dx: ', ave_dx, std_dx)
print('ave_dy, std_dy: ', ave_dy, std_dy)
print('ave_log_dt, std_log_dt: ', ave_log_dt, std_log_dt)
# print('ave_log_dT, std_log_dT: ', ave_log_dT, std_log_dT)


In [ ]:

ave_dx = 1.0750048 
std_dx = 169.44499
ave_dy = -2.5854967
std_dy = 166.78305
ave_log_dt = 2.622117715533515
std_log_dt = 0.7619383343403388

norm_emb_posi_seq_d = (emb_posi_seq_d - np.array([ave_dx, ave_dy])) / np.array([std_dx, std_dy])
norm_emb_time_seq = (np.log(np.stack(emb_time_seq, axis=0)+1.0) - np.array([ave_log_dt])) / np.array([std_log_dt])
# norm_abs_emb_time = (np.log(np.stack(abs_emb_time, axis=0)+1.0) - np.array([ave_log_dT])) / np.array([std_log_dT])
# norm_abs_emb_time_seq = np.stack(abs_emb_time_seq, axis=0)
# combined = np.concatenate([emb_posi_seq, norm_emb_posi_seq_d, norm_emb_time_seq, norm_abs_emb_time, norm_abs_emb_time_seq], axis=-1)
combined = np.concatenate([norm_emb_posi_seq_d, norm_emb_time_seq], axis=-1)
print('combined.shape: ', combined.shape)
print('mask_.shape: ', mask_.shape)
print(len(mask_[100])-sum(mask_[100]))
# print(mask_[100])

f = open('v_thre_time_seq_new.json', 'w')
json.dump(v_thre_time_seq, f, ensure_ascii=False)
f.close()

f = open('v_thre_posi_seq_new.json', 'w')
json.dump(v_thre_posi_seq, f, ensure_ascii=False)
f.close()

In [ ]:
import pandas as pd
from train_trajectory import TrajectoryProcessor
import matplotlib.pyplot as plt

all_dx = []
all_dy = []
all_dt = []

# trace_count = 0
d_list = []
ds_dt_count = 0

for i in range(mask_.shape[0]):
    actual_len = len(mask_[i]) - mask_[i].sum().item()
    
    for j in range(1,actual_len):
        x = emb_posi_seq_d[i, j, 0]
        y = emb_posi_seq_d[i, j, 1]
        all_dx.append(x)
        all_dy.append(y)
        all_dt.append(emb_time_seq[i][j, 0])
        
        d = np.sqrt(x**2 + y**2)
        d_list.append(d)
        # if d < 1e-5:
        #     ds_dt_count += 1
        #     break

# print(ds_dt_count)
# print(ds_dt_count/mask_.shape[0])

d_list = np.array(d_list, dtype=np.float32)
print(np.max(d_list))
print(np.min(d_list))

zeros_ratio = np.mean(d_list < 1e-5)
print(zeros_ratio)

d_thr = d_list[d_list < 1e-5]
t_thr = np.array(all_dt)[d_list < 1e-5]
print(t_thr)
print(len(t_thr))
print(np.min(t_thr))

print(len(d_thr))

custom_bins = [i*1e-4 for i in range(0, 2000)] # 从0到1000，每10一个区间
hist, bin_edges = np.histogram(d_list, bins=custom_bins)
print(f"频数: {hist}")
print(f"边界: {bin_edges}")
plt.plot(bin_edges[:-1], hist)
plt.show()


In [ ]:
# comp_posi = refined_posi_seq[:100]

# track_id = []
# point_order = []
# longitude = []
# latitude = []

# track_count = 0

# for posi in comp_posi:
#     track_count += 1
#     for i in range(len(posi)):
#         track_id.append(track_count)
#         point_order.append(i)
#         longitude.append(posi[i][0])
#         latitude.append(posi[i][1])
        
# df = pd.DataFrame({'track_id': track_id, 'point_order': point_order, 'longitude': longitude, 'latitude': latitude})
# df.to_csv(f'traj_gene_test.csv', index=False, encoding='utf-8')

import json
f = open(f'./training/DDPM_trajectory_v_pre_7/traj_gene.json', 'r')
# f = open(f'traj_gene.json', 'r')
gene_data = json.load(f)
f.close()

track_id = []
point_order = []
longitude = []
latitude = []

for i in range(20):
    one_track = gene_data[i]
    for j in range(len(one_track)):
        track_id.append(i+1)
        point_order.append(j)
        longitude.append(one_track[j][0])
        latitude.append(one_track[j][1])

df = pd.DataFrame({'track_id': track_id, 'point_order': point_order, 'longitude': longitude, 'latitude': latitude})
df.to_csv(f'traj_ge_test_ddpm7.csv', index=False, encoding='utf-8')



for i in range(20):

    count_abnormal = 0
    v_list = []
    for j in range(1, len(gene_data[i])):
        pre_time = gene_data[i][j-1][2]
        pre_lon= gene_data[i][j-1][0]
        pre_lat = gene_data[i][j-1][1]
        
        time = gene_data[i][j][2]
        lon, lat = gene_data[i][j][:2]
        
        d_lon = lon - pre_lon
        d_lat = lat - pre_lat
        # ave_lat = (lat + pre_lat) / 2
        ave_lat = pre_lat
        
        lat_ave_rad = math.radians(ave_lat)
        d_lon_rad = math.radians(d_lon)
        d_lat_rad = math.radians(d_lat)
        
        dx = R * d_lon_rad * math.cos(lat_ave_rad)
        dy = R * d_lat_rad
        dt = time - pre_time
        
        v = math.sqrt(dx*dx + dy*dy) / (dt+1e-6)
        v_list.append(v)
    print(v_list)

In [2]:
import json
from training.DDPM_trajectory_v_pre_15.modules import *
import numpy as np
import pandas as pd
from train_trajectory import TrajectoryProcessor
import inspect
from matplotlib import pyplot as plt
import joblib
from tqdm import tqdm

f = open('./filtered_trips_posi_seq.json', 'r')
trips_posi_seq = json.load(f)
f.close()

f = open('./filtered_trips_time_seq.json', 'r')
trips_time_seq = json.load(f)
f.close()

emb_posi_seq, emb_posi_seq_d, emb_time_seq, start_time_emb, start_end_utm, mask_ = construct_features(trips_time_seq, trips_posi_seq, 120)
# emb_posi_seq, emb_posi_seq_d, emb_time_seq, start_time_emb, mask_ = construct_features(trips_time_seq, trips_posi_seq, 120)
# # emb_posi_seq, emb_posi_seq_d, emb_time_seq, abs_emb_time_seq, mask_ = construct_features(time_seq[:10], posi_seq[:10], 520)
# emb_posi_seq, emb_posi_seq_d, emb_time_seq, abs_emb_time, abs_emb_time_seq, start_time_emb, mask_ = construct_features(trip_time_seq, trip_posi_seq, 120)


# all_dx = []
# all_dy = []
# all_dt = []

# # trace_count = 0
# for i in range(mask_.shape[0]):
#     actual_len = len(mask_[i]) - mask_[i].sum().item()
    
#     ds_dt_count = 0
#     for j in range(1,actual_len):
#         x = emb_posi_seq_d[i, j, 0]
#         y = emb_posi_seq_d[i, j, 1]
#         all_dx.append(x)
#         all_dy.append(y)
#         all_dt.append(emb_time_seq[i][j, 0])

# bins = 20  # 分成10个等宽区间
# counts, bin_edges = np.histogram(all_dt, bins=bins)
# print('len(all_dt): ', len(all_dt))
# print('np.mean(all_dt): ', np.mean(np.array(all_dt)))

# print(f"区间边界: {bin_edges}")
# print(f"各区间计数: {counts}")

# # 可视化
# plt.hist(all_dt, bins=20, edgecolor='black')
# plt.xlabel('Value')
# plt.ylabel('Frequency')
# plt.title('Equal Width Binning')
# plt.show()

# ddpm8 采用的是eps=0.5的参数
# traj_proc = TrajectoryProcessor(eps=0.5)

# # ddpm9第一次是采用的是eps=0.5的参数，没训练完就停止了
# #  ddpm9第二次采用的是eps=0.1的参数
# traj_proc = TrajectoryProcessor(eps=0.1)
# # fit的时候要拉成一个一维长向量
# # transforme的时候要保持原来的形状，最后输出的feature也是和输入形状一样的
# traj_proc.fit(np.array(all_dx), np.array(all_dy), np.array(all_dt))

traj_proc = joblib.load('./d_t_quantile_scaler.pkl')

lengths = np.array([len(seq)-sum(seq) for seq in mask_])
print(lengths[:100])
print('np.mean(lengths): ', np.mean(lengths))

sorted_indices = np.argsort(lengths)
# np.argsort返回的是下标，即把lengths数组从小到大排序后对应的原始下标。sorted_indices[0]是最短轨迹的下标，sorted_indices[-1]是最长轨迹的下标。
lengths_sorted = lengths[sorted_indices]
print('lengths_sorted: ', lengths_sorted[:10])

print(sorted_indices[0])
# print(norm_emb_time_seq[sorted_indices[0]])

# 去重后95%分位数对应的轨迹长度比不去重的更长，实际上重复长度的轨迹应该计算数量。去重后0.95分位数对应的轨迹长度是10349，而不去重的0.95分位数对应的轨迹长度是5204。
i = int(len(lengths)*0.95)+1
print('95%分位数对应的轨迹长度: ', lengths_sorted[i])
length_95 = lengths_sorted[i]
# print('length_95: ', length_95)

values, counts = np.unique(lengths_sorted, return_counts=True)
# plt.bar(values, counts)

new_values = []
new_counts = []

for value, count in zip(values, counts):
    if value <= length_95:
        new_values.append(value)
        new_counts.append(count)
new_values = np.array(new_values)
new_counts = np.array(new_counts)

# plt.plot(new_values, new_counts)
# plt.show()

probabilities = new_counts / new_counts.sum()
print('sum(probabilities): ', sum(probabilities))

# diffusion = Diffusion(noise_steps=2000, beta_start=1e-4, beta_end=0.02, channels=120, emb_dim=5, device='cuda:0')
diffusion = Diffusion(noise_steps=2000, beta_start=1e-4, beta_end=0.02, channels=120, emb_dim=4, device='cuda:0')

# model = Pred_model(c_in=120, c_out=120, emb_dim=512, first_tim_dim=8, first_pos_dim=40, length_dim=128, device="cuda:0")
model = Pred_model(c_in=120, c_out=120, emb_dim=512, first_tim_dim=8, first_pos_dim=40, length_dim=128, device="cuda:0")
checkpoint = torch.load('./training/DDPM_trajectory_v_pre_15/ckpt_1499.pt')
model.load_state_dict(checkpoint)
model.to('cuda:0')

# x_test = torch.randn(10, 120, 5).float().to('cuda:0')
# t_test = torch.ones(10).long().float().to('cuda:0')
# start_time_ = torch.from_numpy(start_time_emb[:10, ...]).float().to('cuda:0')
# od_posi_ = torch.from_numpy(emb_posi_seq[:10, :2, :]).float().to('cuda:0')
# padding_mask = torch.from_numpy(mask_[:10, :]).bool().to('cuda:0')
# predicted_v = model(x_test, t_test, padding_mask, start_time_, od_posi_)
# print(predicted_v)

track_id = []
point_order = []
longitude = []
latitude = []

track_id_real = []
point_order_real = []
longitude_real = []
latitude_real = []

track_count = 0

dec_trajectories = []

n = 128
batch_number = emb_posi_seq.shape[0] // n
# batch_number = 2

print(batch_number)

for ii in tqdm(range(batch_number)):
    start_time_ = torch.from_numpy(start_time_emb[n*ii:n*(ii+1), ...]).float().to('cuda:0')
    od_posi_ = torch.from_numpy(emb_posi_seq[n*ii:n*(ii+1), :2, :]).float().to('cuda:0')
    padding_mask = torch.from_numpy(mask_[n*ii:n*(ii+1), :]).bool().to('cuda:0')

    
    x_gen = diffusion.sample_with_mask(model, n, padding_mask, start_time_, od_posi_)


    ddx, ddy, ddt, is_stationary = traj_proc.inverse_transformer(x_gen)
    

    comp_time = trips_time_seq[n*ii:n*(ii+1)]
    comp_posi = trips_posi_seq[n*ii:n*(ii+1)]
    # print(comp_posi)
    # print(comp_time)
    
    pad_mask = padding_mask.cpu().numpy()
    
    for i in range(len(pad_mask)):
        track_count += 1
        
        actual_len = len(pad_mask[i]) - sum(pad_mask[i])
        # print(f"生成轨迹 {i} 的实际长度: {actual_len} 步")
        lon = comp_posi[i][0][0]
        lat = comp_posi[i][0][1]
        tim = comp_time[i][0]
        # print(f"起始位置纬度: {lat} 度, 经度: {lon} 度, 时间: {tim:.2f} 秒")
        
        one_traj = [[lon, lat, tim]]
        one_time = [tim]
        
        track_id.append(track_count)
        point_order.append(0)
        longitude.append(lon)
        latitude.append(lat)
        
        track_id_real.append(track_count)
        point_order_real.append(0)
        longitude_real.append(lon)
        latitude_real.append(lat)
        
        for j in range(1, actual_len):
            track_id_real.append(track_count)
            point_order_real.append(j)
            longitude_real.append(comp_posi[i][j][0])
            latitude_real.append(comp_posi[i][j][1])
            
            
            dx = ddx[i][j]
            dy = ddy[i][j]
            dt = ddt[i][j]
            
            rad_lat = math.radians(lat)

            d_lat_rad = (dy / R)*180 / math.pi
            d_lon_rad = (dx / (R * math.cos(rad_lat)))*180 / math.pi
            
            lat += d_lat_rad
            lon += d_lon_rad
            
            tim += dt
            one_traj.append([lon, lat, tim])
            # one_traj.append([lon, lat])
            one_time.append(tim)
            
            track_id.append(track_count)
            point_order.append(j)
            longitude.append(lon)
            latitude.append(lat)
            
            # print(f"位置增量: ({x_gen[i, j, 0].item():.2f}, {x_gen[i, j, 1].item():.2f}) 米, 时间增量: {x_gen[i, j, 2].item():.2f} 秒")
        dec_trajectories.append(one_traj)
        # print(one_traj)
        # print(one_time)
        
# df = pd.DataFrame({'track_id': track_id, 'point_order': point_order, 'longitude': longitude, 'latitude': latitude})
# df.to_csv(f'traj_gene_ddpm15.csv', index=False, encoding='utf-8')

# df_real = pd.DataFrame({'track_id': track_id_real, 'point_order': point_order_real, 'longitude': longitude_real, 'latitude': latitude_real})
# df_real.to_csv(f'traj_gene_ddpm15_real.csv', index=False, encoding='utf-8')

f = open('traj_gene_ddpm15_all.json', 'w', encoding='utf-8')
json.dump(dec_trajectories, f, ensure_ascii=False)
f.close()

去掉长度小于5的轨迹数量: 144828, 144828, 144828
[ 41  26 120 120  30  53 120  71 120 112 120 120 120 120  78 120 120 120
 100 120 120  75  37 119 114 102 106 120  73 120  95  34  46  62 120 120
  24  94  74  38  34  42  24  46  41  34  33  33  46  42  21  39  44  43
  32  40  29  34  40  43  43  30  40  41  26  40  47  21  33  42  29  34
  44  39  23  42  30  43  20  43  43  46  40  41  34  37  21  24  24  21
  24  25  36  29  35  34  24  31  20  24]
np.mean(lengths):  38.71110558731737
lengths_sorted:  [20 20 20 20 20 20 20 20 20 20]
56895
95%分位数对应的轨迹长度:  120
sum(probabilities):  1.0000000000000007
tensor(1.0000, device='cuda:0')
tensor(0., device='cuda:0')
tensor(0., device='cuda:0')
tensor(1., device='cuda:0')
1131


645it [00:46, 13.95it/s]


KeyboardInterrupt: 

In [ ]:
import json

f = open('./training/DDPM_trajectory_v_pre_15/training_process_v_pre_1500_epochs.json', 'r', encoding='utf-8')
data = json.load(f)
f.close()

loss_min = 1000000.0
min_key = None
for key in data.keys():
    loss = data[key][0]
    if loss < loss_min:
        loss_min = loss
        min_key = key
print('loss_min: ', loss_min)
print('min_key: ', min_key)
